# HyperParameterOptimisation for Approval Predict

## Objective:
Complete hyperprarameter tuning on the features of 'years_employed', 'loan_amount' and 'income'. This is due to test model performance for points, credit score feature  100% prediction accuracy.

## Inputs
outputs/datasets/collection/loan_approved.csv

## Additional information
There are no outputs for this notebook. This notebook was created to complete hyperparameter optimisation on the best 3 ML models found on the previous notebooks. Loan to income was removed as a feature.

## Imports and required libraries

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    classification_report, confusion_matrix,
    precision_score, recall_score, f1_score, make_scorer
)
from sklearn.pipeline import Pipeline
from sklearn.ensemble import (
    RandomForestClassifier, AdaBoostClassifier
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
import warnings
warnings.filterwarnings("ignore")

## Update Working Directory

We want to make the parent of the current directory the new current directory and confirm this.

In [ ]:
current_dir = os.getcwd()
current_dir

In [ ]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

In [ ]:
current_dir = os.getcwd()
current_dir

## Load data

Load the raw dataset and change the target variable from a boolean to an integer. Created variable reduced_features to use for hyperparameter optimisation.

In [ ]:
root = os.getcwd()
file_path = Path(root) / "outputs" / "datasets" / "collection" / "loan_approval.csv"
df = pd.read_csv(file_path).drop(['name', 'city'], axis=1)

df['loan_approved'] = df['loan_approved'].astype(int)

reduced_features = ['years_employed', 'loan_amount', 'income']
X = df[reduced_features].copy()
y = df['loan_approved'].copy()

X.head(3), y.head(3)

## Split train and test sets

The data is split into train (80%) and test sets (20%).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print("Train/Test Shapes:")
X_train.shape, y_train.shape, X_test.shape, y_test.shape

## Handle Class Imbalance

 Class imbalance handling was initially applied across all machine learning models. However, this approach led to reduced performance on the test sets for both the RandomForest and AdaBoost classifiers, so it was subsequently removed from those models. It was retained for the XGBoost classifier, as it did not negatively impact its results.

In [ ]:
smote = SMOTE(sampling_strategy='minority', random_state=42)

X_train_bal, y_train_bal = smote.fit_resample(X_train, y_train)

print("After SMOTE:", X_train_bal.shape, y_train_bal.shape)

y_train_bal.value_counts().plot(kind='bar', title='Balanced Loan Approved Distribution')
plt.show()

## ML Pipeline

Each classifier is wrapped within a Pipeline object using the PipelineClf() function. This structure allows consistent interaction with GridSearchCV for hyperparameter optimization and makes it easier to compare multiple models.

In [ ]:
def PipelineClf(model):
    return Pipeline([
        ("model", model)
    ])

In [ ]:
def PipelineClf(model):
    """
    Builds a pipeline for hyperparameter optimisation.
    """
    return Pipeline([
        ("model", model),
    ])

## HyperParameter Optimisation

A custom class, HyperparameterOptimizationSearch, was created by the CodeInstitute to perform grid searches across selected parameter spaces for each model.

The grid search was evaluated using cross-validation (cv=5).

The scoring metric used was recall (pos_label=1), as the business objective prioritised identifying as many “Approved” applicants as possible.

In [ ]:
class HyperparameterOptimizationSearch:

    def __init__(self, models, params):
        self.models = models
        self.params = params
        self.keys = models.keys()
        self.grid_searches = {}

    def fit(self, X, y, cv, n_jobs, verbose=1, scoring=None, refit=False):
        for key in self.keys:
            print(f"\nRunning GridSearchCV for {key} \n")

            model = PipelineClf(self.models[key])
            params = self.params[key]
            gs = GridSearchCV(model, params, cv=cv, n_jobs=n_jobs,
                              verbose=verbose, scoring=scoring, )
            gs.fit(X, y)
            self.grid_searches[key] = gs

    def score_summary(self, sort_by='mean_score'):
        def row(key, scores, params):
            d = {
                'estimator': key,
                'min_score': min(scores),
                'max_score': max(scores),
                'mean_score': np.mean(scores),
                'std_score': np.std(scores),
            }
            return pd.Series({**params, **d})

        rows = []
        for k in self.grid_searches:
            params = self.grid_searches[k].cv_results_['params']
            scores = []
            for i in range(self.grid_searches[k].cv):
                key = "split{}_test_score".format(i)
                r = self.grid_searches[k].cv_results_[key]
                scores.append(r.reshape(len(params), 1))

            all_scores = np.hstack(scores)
            for p, s in zip(params, all_scores):
                rows.append((row(k, s, p)))

        df = pd.concat(rows, axis=1).T.sort_values([sort_by], ascending=False)
        columns = ['estimator', 'min_score',
                   'mean_score', 'max_score', 'std_score']
        columns = columns + [c for c in df.columns if c not in columns]
        return df[columns], self.grid_searches

The 2 functions below were taken from the CodeInstitute to provide reporting of performance metrics (accuracy, precision, recall, and F1-score). They will give a comparison between the classifiers AdaBoost, RandomForest, and XGBoost.

In [ ]:
def confusion_matrix_and_report(X, y, pipeline, label_map):
    prediction = pipeline.predict(X)

    print('---  Confusion Matrix  ---')
    print(pd.DataFrame(
        confusion_matrix(y_true=prediction, y_pred=y),
        columns=[["Actual " + sub for sub in label_map]],
        index=[["Prediction " + sub for sub in label_map]]
    ))
    print("\n")

    print('---  Classification Report  ---')
    print(classification_report(y, prediction, target_names=label_map), "\n")

def clf_performance(X_train, y_train, X_test, y_test, pipeline, label_map):
    print("#### Train Set #### \n")
    confusion_matrix_and_report(X_train, y_train, pipeline, label_map)

    print("#### Test Set ####\n")
    confusion_matrix_and_report(X_test, y_test, pipeline, label_map)

First round of hyperparameter optimisation using AdaBoostClassifier

In [ ]:
model = {"AdaBoostClassifier": AdaBoostClassifier(random_state=42)}

https://www.youtube.com/watch?v=JmXnztjULnQ 
Tune its key parameters (such as n_estimators and learning_rate) to improve its generalisation.
https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.AdaBoostClassifier.html
Scikit-learn details default parameters of 50 for n_estimators and 1 for learning rate. I adjusted these values

In [ ]:
params_search_ada = {
    "AdaBoostClassifier": {
        "model__n_estimators": [50,25,80,150],
        "model__learning_rate": [1,0.1, 2]
    }
}

n_estimators: Number of weak learners to train iteratively. How many small trees the model builds and combines to make predictions.
learning_rate: It contributes to the weights of weak learners. How strongly each tree changes the model’s final prediction.

https://www.datacamp.com/tutorial/adaboost-classifier-python?dc_referrer=https%3A%2F%2Fwww.google.com%2F

In [1]:
search = HyperparameterOptimizationSearch(models=model, params=params_search_ada)
search.fit(X_train, y_train,
           scoring=make_scorer(recall_score, pos_label=1),
           cv=5, n_jobs=-1)

NameError: name 'HyperparameterOptimizationSearch' is not defined

In [ ]:
extensive_grid_search_summary, extensive_grid_search_pipelines = search.score_summary(sort_by='mean_score')

best_model = extensive_grid_search_summary.iloc[0,0]
best_parameters = extensive_grid_search_pipelines[best_model].best_params_

classification_pipeline = extensive_grid_search_pipelines[best_model].best_estimator_

best_parameters

In [ ]:
clf_performance(X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                pipeline=classification_pipeline,
                label_map=["Rejected", "Approved"])

In [ ]:
params_search_ada = {
    "AdaBoostClassifier": {
        "model__n_estimators": [10,25,80,150],
        "model__learning_rate": [1,0.5, 2]
    }
}

In [ ]:
search = HyperparameterOptimizationSearch(models=model, params=params_search_ada)
search.fit(X_train, y_train,
           scoring=make_scorer(recall_score, pos_label=1),
           cv=5, n_jobs=-1)

In [ ]:
extensive_grid_search_summary, extensive_grid_search_pipelines = search.score_summary(sort_by='mean_score')

best_model = extensive_grid_search_summary.iloc[0,0]
best_parameters = extensive_grid_search_pipelines[best_model].best_params_

classification_pipeline = extensive_grid_search_pipelines[best_model].best_estimator_

best_parameters

In [ ]:
clf_performance(X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                pipeline=classification_pipeline,
                label_map=["Rejected", "Approved"])

In [ ]:
ada_best = AdaBoostClassifier(
    learning_rate=1.2,
    n_estimators=15,
    random_state=42
)

In [ ]:
pipeline_ada = Pipeline([
    ('model', ada_best)
])

In [ ]:
pipeline_ada.fit(X_train, y_train)

In [ ]:
y_pred_train = pipeline_ada.predict(X_train)
y_pred_test = pipeline_ada.predict(X_test)

print("#### Train Set ####\n")
print(confusion_matrix(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train, target_names=["Rejected","Approved"]))

print("#### Test Set ####\n")
print(confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test, target_names=["Rejected","Approved"]))

## RandomForestClassifier

In [ ]:
model = {"RandomForestClassifier": RandomForestClassifier(random_state=42)}

In [ ]:
params_search = {
    "RandomForestClassifier": {
        'model__n_estimators': [100,50,140],
        'model__max_depth': [None,4, 15],
        'model__min_samples_split': [2,50],
        'model__min_samples_leaf': [1,50],
        'model__max_leaf_nodes': [None,50],
    },
}

In [ ]:
search = HyperparameterOptimizationSearch(models=model, params=params_search)
search.fit(X_train, y_train,
           scoring=make_scorer(recall_score, pos_label=1),
           cv=5, n_jobs=-1)

In [ ]:
extensive_grid_search_summary, extensive_grid_search_pipelines = search.score_summary(sort_by='mean_score')

best_model = extensive_grid_search_summary.iloc[0,0]
best_parameters = extensive_grid_search_pipelines[best_model].best_params_

classification_pipeline = extensive_grid_search_pipelines[best_model].best_estimator_

best_parameters

In [ ]:
clf_performance(X_train=X_train, y_train=y_train,
                X_test=X_test, y_test=y_test,
                pipeline=classification_pipeline,
                label_map=["Rejected", "Approved"])

In [ ]:
rf_best = RandomForestClassifier(
    max_depth=6,
    max_leaf_nodes=None,
    min_samples_leaf=1,
    min_samples_split=2,
    n_estimators=100,
    class_weight="balanced",
    random_state=42
)

In [ ]:
pipeline_rf = Pipeline([
    ('model', rf_best)
])

In [ ]:
pipeline_rf.fit(X_train, y_train)

In [ ]:
clf_performance(
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    pipeline=pipeline_rf,
    label_map=["Rejected", "Approved"]
)

In [ ]:
y_pred_train = pipeline_rf.predict(X_train)
y_pred_test = pipeline_rf.predict(X_test)

print("#### Train Set ####\n")
print(confusion_matrix(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train, target_names=["Rejected","Approved"]))

print("#### Test Set ####\n")
print(confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test, target_names=["Rejected","Approved"]))

## XGBClassifier

In [ ]:
model = {"XGBClassifier": XGBClassifier(random_state=42)}

In [ ]:
params_search_xgb = {
    "XGBClassifier": {
        "model__n_estimators": [30, 80, 200],
        "model__max_depth": [None, 6, 15],
        "model__learning_rate": [0.1, 0.01, 0.001],
        "model__gamma": [0, 0.1],
    }
}

In [ ]:
search = HyperparameterOptimizationSearch(models=model, params=params_search_xgb)

search.fit(
    X_train_bal, y_train_bal,
    scoring=make_scorer(recall_score, pos_label=1),
    n_jobs=-1, cv=5
)

In [ ]:
extensive_grid_search_summary, extensive_grid_search_pipelines = search.score_summary(sort_by='mean_score')

best_model = extensive_grid_search_summary.iloc[0,0]
best_parameters = extensive_grid_search_pipelines[best_model].best_params_
classification_pipeline = extensive_grid_search_pipelines[best_model].best_estimator_

best_parameters

In [ ]:
clf_performance(
    X_train=X_train_bal, y_train=y_train_bal,
    X_test=X_test, y_test=y_test,
    pipeline=classification_pipeline,
    label_map=["Rejected", "Approved"]
)

In [ ]:
xgb_best = XGBClassifier(
    gamma=0,
    learning_rate=0.2,
    max_depth=None,
    n_estimators=80,
    random_state=42,
    eval_metric='logloss',
)

In [ ]:
pipeline_xgb = Pipeline([
    ('model', xgb_best)
])

In [ ]:
pipeline_xgb.fit(X_train, y_train)

In [ ]:
clf_performance(
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    pipeline=pipeline_xgb,
    label_map=["Rejected", "Approved"]
)

In [ ]:
y_pred_train = pipeline_xgb.predict(X_train)
y_pred_test = pipeline_xgb.predict(X_test)

print("#### Train Set ####\n")
print(confusion_matrix(y_train, y_pred_train))
print(classification_report(y_train, y_pred_train, target_names=["Rejected","Approved"]))

print("#### Test Set ####\n")
print(confusion_matrix(y_test, y_pred_test))
print(classification_report(y_test, y_pred_test, target_names=["Rejected","Approved"]))

## Conclusions

The RandomForest model achieved a recall of 69% for Approved applications on unseen test data, outperforming both AdaBoost and XGBoost.

THe features credit score, loan-to-income ratio, and points were already 100% predictive across both train and test datasets, confirming that they align perfectly with existing business approval rules and do not require optimisation.
The RandomForest model therefore adds value primarily in borderline cases where deterministic criteria are insufficient. Further improvements will likely depend on enhanced data quality or adding new features.

## Next steps
The RainForestClassification will be used in the ML pipeline.